# Tome C1, Chapter 10: Matroids (Python companion)

Exercises first, solutions at the end. A matroid is given to the computer the way it is given in the chapter: a ground set and an *independence oracle*, a function that answers "is this set independent?". With that one function we check the axioms on examples and non-examples, race the greedy algorithm against brute force, schedule jobs, compute duals, hunt for representations of the Fano and non-Fano matroids over small fields, walk on the tree graph, and run Edmonds' matroid intersection algorithm, checked against bipartite matching from Chapter 6.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math
from collections import deque, Counter
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
random.seed(10); rng = np.random.default_rng(10)

## Exercises

**P10.1 (★) Oracles and axioms.** Write a small `Matroid` class from a ground set and an independence oracle, with a rank function computed greedily. Build uniform, graphic, linear (over $\mathbb F_p$), partition and transversal matroids (Proposition 10.2), and check the axioms (I1)–(I3) by brute force. Check also that the matchings of a path do not form a matroid (Exercise 10.4).

**P10.2 (★★) Greed versus brute force.** For random weights on graphic and linear matroids, compare greedy with the best basis found by trying all independent sets. For a set system that fails (I3), build the weights of the proof of Theorem 10.8 and watch greedy lose.

**P10.3 (★) Scheduling.** Solve the instance of Exercise 10.5 greedily and by brute force, then compare on random instances.

**P10.4 (★★) Bases, rank and duality.** Check the basis exchange property (Theorem 10.7), submodularity (Theorem 10.6) and the dual rank formula (Theorem 10.10) on random examples, and check on a random graph that the circuits of $M(G)^*$ are its bonds (Proposition 10.11).

**P10.5 (★★) Fano and non-Fano.** Check the determinants of all $35$ triples of the seven vectors of $\mathbb F_2^3\setminus\{0\}$. Then search exhaustively, in the normal form of the proof of Theorem 10.12, for representations of the Fano and non-Fano matroids over $\mathbb F_2,\mathbb F_3,\mathbb F_5,\mathbb F_7$ (Exercise 10.9). Check also that $U_{2,4}$ is not binary (Exercise 10.8).

**P10.6 (★★) The tree graph.** On a small graph, build the graph whose vertices are the spanning trees, two of them joined when they differ by one exchange, and check that the distance between $T_1$ and $T_2$ is $|T_1\setminus T_2|$ (Exercise 10.11).

**P10.7 (★★★) Matroid intersection.** Implement the augmenting path algorithm for the largest common independent set of two matroids, using the exchange graph of the remark on matroid intersection. Check it against brute force and against Edmonds' min formula on small random pairs, against maximum bipartite matching (Kőnig, Theorem 6.4) for two partition matroids, and use it to find a spanning forest of a graph whose edges have different colours, using each colour at most once (a *rainbow* forest).

## Solutions

### P10.1 Oracles and axioms

The rank of $X$ is computed greedily: by Proposition 10.4, every maximal independent subset of $X$ has the same size, so scanning $X$ and keeping what stays independent finds one. For a non-matroid this "rank" is only a lower bound, which is fine since we only use it on matroids.

In [ ]:
class Matroid:
    def __init__(self, E, indep, name=''):
        self.E, self.indep, self.name = list(E), indep, name
    def rank(self, X):
        I = []
        for x in X:
            if self.indep(I + [x]):
                I.append(x)
        return len(I)
    def independent_sets(self):
        return [frozenset(S) for k in range(len(self.E) + 1)
                for S in itertools.combinations(self.E, k) if self.indep(list(S))]

def check_axioms(M):
    # (I1), (I2) and (I3) by brute force; returns a counterexample to (I3) if any.
    fam = set(M.independent_sets())
    if frozenset() not in fam:
        return 'I1 fails'
    for S in fam:
        for x in S:
            if S - {x} not in fam:
                return f'I2 fails at {set(S)}'
    for A in fam:
        for B in fam:
            if len(A) < len(B) and not any(A | {x} in fam for x in B - A):
                return f'I3 fails: A={set(A)}, B={set(B)}'
    return 'matroid'

def rank_mod_p(vectors, p):
    rows = [list(v) for v in vectors]; r = 0
    ncol = len(rows[0]) if rows else 0
    for c in range(ncol):
        piv = next((i for i in range(r, len(rows)) if rows[i][c] % p), None)
        if piv is None:
            continue
        rows[r], rows[piv] = rows[piv], rows[r]
        inv = pow(rows[r][c], p - 2, p)
        rows[r] = [v * inv % p for v in rows[r]]
        for i in range(len(rows)):
            if i != r and rows[i][c] % p:
                f = rows[i][c]; rows[i] = [(a - f * b) % p for a, b in zip(rows[i], rows[r])]
        r += 1
    return r

def uniform(k, n):
    return Matroid(range(n), lambda S: len(S) <= k, f'U_{k},{n}')
def graphic(G):
    edges = list(G.edges())
    def indep(S):
        # union-find: S is a forest iff no edge joins two vertices already connected
        parent = {}
        def find(v):
            while parent.get(v, v) != v:
                v = parent[v]
            return v
        for i in S:
            a, b = find(edges[i][0]), find(edges[i][1])
            if a == b:
                return False
            parent[a] = b
        return True
    return Matroid(range(len(edges)), indep, 'graphic')
def linear(vectors, p):
    return Matroid(range(len(vectors)), lambda S: rank_mod_p([vectors[i] for i in S], p) == len(S), f'linear over F_{p}')
def partition(blocks, caps):
    where = {x: b for b, B in enumerate(blocks) for x in B}
    E = [x for B in blocks for x in B]
    return Matroid(E, lambda S: all(c <= caps[b] for b, c in Counter(where[x] for x in S).items()), 'partition')
def transversal(E, sets):
    # S is independent iff S can be matched into distinct sets (Hopcroft-Karp from networkx)
    def indep(S):
        if not S:
            return True
        B = nx.Graph(); B.add_nodes_from((('e', x) for x in S))
        B.add_edges_from((('e', x), ('s', j)) for x in S for j, A in enumerate(sets) if x in A)
        m = nx.bipartite.maximum_matching(B, top_nodes=[('e', x) for x in S])
        return sum(1 for k in m if k[0] == 'e') == len(S)
    return Matroid(E, indep, 'transversal')
def path_matchings():
    # path u-v-w-z with edges a=uv, b=vw, c=wz
    ends = {'a': 'uv', 'b': 'vw', 'c': 'wz'}
    return Matroid('abc', lambda S: len(set(''.join(ends[e] for e in S))) == 2 * len(S), 'matchings of P4')

G = nx.petersen_graph()
vecs = [tuple(random.randrange(3) for _ in range(3)) for _ in range(7)]
examples = [uniform(2, 5), graphic(nx.complete_graph(5)), linear(vecs, 3),
            partition([[0, 1, 2], [3, 4], [5, 6, 7]], [1, 2, 2]),
            transversal(range(7), [{0, 1, 2}, {2, 3}, {3, 4, 5}, {5, 6, 0}]), path_matchings()]
for M in examples:
    print(f'{M.name:22s} |E|={len(M.E):2d}  rank={M.rank(M.E)}  #independent={len(M.independent_sets()):4d}  ->', check_axioms(M))

Five matroids and one impostor: the matchings of the path fail (I3) with $A=\{b\}$, $B=\{a,c\}$, exactly as in Exercise 10.4. The graphic matroid of $K_5$ has rank $4$ (spanning trees have $4$ edges) and $1+10+45+110+125=291$ forests; the last number is Cayley's $5^{5-2}$ trees.

### P10.2 Greed versus brute force

In [ ]:
def greedy(M, w):
    I = []
    for x in sorted(M.E, key=lambda x: -w[x]):
        if w[x] > 0 and M.indep(I + [x]):
            I.append(x)
    return I
def best(M, w):
    return max(M.independent_sets(), key=lambda S: sum(w[x] for x in S))

trials = 0
for t in range(40):
    G = nx.gnp_random_graph(6, 0.6, seed=t)
    for M in [graphic(G), linear([tuple(random.randrange(5) for _ in range(3)) for _ in range(8)], 5)]:
        w = {x: random.randint(-3, 20) for x in M.E}
        g, b = greedy(M, w), best(M, w)
        assert sum(w[x] for x in g) == sum(w[x] for x in b)
        trials += 1
print(trials, 'random instances: greedy is always optimal on matroids')

def defeat_greed(M):
    # Proof of Theorem 10.8: A, B independent, |A| < |B|, no x in B - A with A + x independent.
    fam = set(M.independent_sets())
    for A in fam:
        for B in fam:
            if len(A) < len(B) and not any(A | {x} in fam for x in B - A):
                a, b, eps = len(A), len(B), 1 / (2 * len(A) + 2)
                w = {x: (1 + eps if x in A else 1 if x in B else 0) for x in M.E}
                return A, B, w
for M in [path_matchings(),
          Matroid(range(5), lambda S: len(S) <= 1 or set(S) <= {0, 1, 2} or set(S) <= {2, 3, 4}, 'two triangles glued')]:
    A, B, w = defeat_greed(M)
    g, b = greedy(M, w), best(M, w)
    print(f'{M.name}: A={set(A)}, B={set(B)}; greedy {sorted(g)} weighs {sum(w[x] for x in g):.3f}, optimum {sorted(b)} weighs {sum(w[x] for x in b):.3f}')

The weights of the proof, $1+\varepsilon$ on $A$, $1$ on $B\setminus A$ and $0$ elsewhere, make greedy take $A$ first and then lock it out of $B$. This converse is what makes the greedy theorem a *characterisation*: matroids are exactly the hereditary set systems where greed is never punished.

### P10.3 Scheduling

In [ ]:
def feasible(jobs, d):
    # at most t jobs with deadline <= t, for every t (Example 10.3)
    ds = sorted(d[j] for j in jobs)
    return all(ds[k] >= k + 1 for k in range(len(ds)))
def scheduling(d):
    return Matroid(range(len(d)), lambda S: feasible(S, d), 'scheduling')

d = [2, 1, 2, 1, 3, 3]; w = dict(enumerate([10, 9, 8, 7, 6, 5]))
M = scheduling(d)
print(check_axioms(M))
g = greedy(M, w)
print('greedy keeps jobs', [j + 1 for j in g], 'reward', sum(w[j] for j in g),
      '| brute force', sum(w[j] for j in best(M, w)))
print('timetable (day: job):', {k + 1: j + 1 for k, j in enumerate(sorted(g, key=lambda j: d[j]))})
for t in range(200):
    n = random.randint(3, 9)
    d = [random.randint(1, n) for _ in range(n)]; w = {j: random.randint(1, 30) for j in range(n)}
    M = scheduling(d)
    assert sum(w[j] for j in greedy(M, w)) == sum(w[j] for j in best(M, w))
print('200 random instances agree')

Greedy keeps jobs $1,2,5$ for $25$, done in the order $2,1,5$. The test "the $k$-th smallest deadline is at least $k$" is the condition $N_A(t)\le t$ of Exercise 10.5, read from the other side.

### P10.4 Bases, rank and duality

In [ ]:
def bases(M):
    fam = M.independent_sets(); r = max(len(S) for S in fam)
    return [S for S in fam if len(S) == r]
def dual(M):
    Bs = bases(M); E = frozenset(M.E)
    co = {E - B for B in Bs}
    return Matroid(M.E, lambda S: any(set(S) <= C for C in co), M.name + '*')
def circuits(M):
    fam = set(M.independent_sets()); out = []
    for k in range(1, len(M.E) + 1):
        for S in itertools.combinations(M.E, k):
            S = frozenset(S)
            if S not in fam and all(S - {x} in fam for x in S):
                out.append(S)
    return out

for t in range(5):
    G = nx.gnp_random_graph(6, 0.55, seed=20 + t)
    if not nx.is_connected(G):
        continue
    M = graphic(G); Bs = bases(M); Bset = set(Bs)
    # exchange: for x in B1 - B2 there is y in B2 - B1 with B1 - x + y a basis
    assert all(any((B1 - {x}) | {y} in Bset for y in B2 - B1) for B1 in Bs for B2 in Bs for x in B1 - B2)
    subsets = [frozenset(S) for k in range(len(M.E) + 1) for S in itertools.combinations(M.E, k)]
    sample = random.sample(subsets, 60)
    assert all(M.rank(list(A | B)) + M.rank(list(A & B)) <= M.rank(list(A)) + M.rank(list(B)) for A in sample for B in sample)
    D = dual(M); rE = M.rank(M.E)
    assert all(D.rank(list(A)) == len(A) - rE + M.rank([x for x in M.E if x not in A]) for A in sample)
    edges = list(G.edges())
    bonds = []
    for k in range(1, len(edges) + 1):
        for S in itertools.combinations(range(len(edges)), k):
            H = G.copy(); H.remove_edges_from(edges[i] for i in S)
            if not nx.is_connected(H) and not any(set(b) <= set(S) for b in bonds):
                bonds.append(S)
    assert {frozenset(b) for b in bonds} == set(circuits(D))
    print(f'graph with {G.number_of_nodes()} vertices, {len(edges)} edges: {len(Bs)} spanning trees, '
          f'{len(circuits(M))} cycles, {len(bonds)} bonds = circuits of the dual')

All four statements hold on every example. The bond list was built without any matroid language (minimal edge sets whose removal disconnects), and it coincides with the circuits of the dual computed from complements of spanning trees.

### P10.5 Fano and non-Fano

In [ ]:
V = {1: (1, 0, 0), 2: (0, 1, 0), 3: (0, 0, 1), 4: (1, 1, 0), 5: (1, 0, 1), 6: (0, 1, 1), 7: (1, 1, 1)}
LINES = [{1, 2, 4}, {1, 3, 5}, {2, 3, 6}, {1, 6, 7}, {2, 5, 7}, {3, 4, 7}, {4, 5, 6}]
dets = Counter()
for t in itertools.combinations(V, 3):
    dets[(round(np.linalg.det(np.array([V[i] for i in t]))), set(t) in LINES)] += 1
print('(integer determinant, is a line): count ->', dict(dets))

def det3(u, v, w, p):
    return (u[0] * (v[1] * w[2] - v[2] * w[1]) - u[1] * (v[0] * w[2] - v[2] * w[0]) + u[2] * (v[0] * w[1] - v[1] * w[0])) % p

def represents(vec, lines, p):
    # the dependent triples over F_p are exactly the given lines (pairs are independent since the points are distinct)
    return all((det3(*(vec[i] for i in t), p) == 0) == (set(t) in lines) for t in itertools.combinations(range(1, 8), 3))

def proj_points(p):
    # one non-zero vector on each line through the origin of F_p^3
    return [v for v in itertools.product(range(p), repeat=3) if any(v) and v[next(i for i in range(3) if v[i])] == 1]

def search(lines, p):
    # Normal form of the proof of Theorem 10.12, with each vector rescaled (which does not change the matroid):
    # v1, v2, v3 standard, v4=(1,b,0), v5=(1,0,d), v6=(0,1,f), and v7 any of the p^2+p+1 points.
    found = 0
    P = proj_points(p)
    for b, d, f in itertools.product(range(1, p), repeat=3):
        for v7 in P:
            vec = {1: (1, 0, 0), 2: (0, 1, 0), 3: (0, 0, 1), 4: (1, b, 0), 5: (1, 0, d), 6: (0, 1, f), 7: v7}
            if len(set(vec.values())) == 7 and represents(vec, lines, p):
                found += 1
    return found

NONFANO = LINES[:-1]
for p in [2, 3, 5, 7]:
    print(f'F_{p}: normal forms representing Fano: {search(LINES, p):3d}   non-Fano: {search(NONFANO, p):3d}')

The integer determinants are $0$ on six lines, $-2$ on the line $456$, and $\pm1$ on the other $28$ triples, as used in the proof of Theorem 10.12. The search confirms the theorem and Exercise 10.9: the Fano matroid has representations only over $\mathbb F_2$, the non-Fano matroid only over the odd fields. The counts are the numbers of normal forms: for the non-Fano matroid, $b$ and $d$ are free, and then the proof forces $v_7=(1,b,d)$ (lines $347$ and $257$) and $f=\frac db$ (line $167$), giving $(p-1)^2$; for the Fano matroid over $\mathbb F_2$ everything is forced. So up to rescaling the vectors and changing coordinates, the Fano plane has exactly one representation over $\mathbb F_2$ and the non-Fano matroid $(p-1)^2$ over $\mathbb F_p$ (some of which are related by further changes of coordinates).

In [ ]:
U24_over_F2 = [M for M in itertools.combinations(itertools.product(range(2), repeat=2), 4)
               if all(rank_mod_p([u, v], 2) == 2 for u, v in itertools.combinations(M, 2))]
U24_over_F3 = [M for M in itertools.combinations(itertools.product(range(3), repeat=2), 4)
               if all(rank_mod_p([u, v], 3) == 2 for u, v in itertools.combinations(M, 2))]
print('4 pairwise independent vectors in F_2^2:', len(U24_over_F2), '| in F_3^2:', len(U24_over_F3))

No four vectors of $\mathbb F_2^2$ are pairwise independent (there are only three lines through the origin), while $\mathbb F_3^2$ has $4$ lines and $2^4=16$ ways to pick a non-zero vector on each.

### P10.6 The tree graph

In [ ]:
G = nx.Graph([(0, 1), (1, 2), (2, 3), (3, 0), (0, 2), (1, 3), (3, 4), (2, 4)])
M = graphic(G); T = bases(M); idx = {B: i for i, B in enumerate(T)}
TG = nx.Graph(); TG.add_nodes_from(range(len(T)))
for B in T:
    for x in B:
        for y in set(M.E) - B:
            C = (B - {x}) | {y}
            if C in idx:
                TG.add_edge(idx[B], idx[C])
dist = dict(nx.all_pairs_shortest_path_length(TG))
assert all(dist[i][j] == len(T[i] - T[j]) for i in range(len(T)) for j in range(len(T)))
print(f'{len(T)} spanning trees, tree graph connected: {nx.is_connected(TG)}, diameter {nx.diameter(TG)} '
      f'(= n - 1 = {G.number_of_nodes() - 1} here), distance = |T1 - T2| for all pairs')

### P10.7 Matroid intersection

For a common independent set $I$, the *exchange graph* has the elements as vertices; for $y\in I$ and $x\notin I$ there is an arc $y\to x$ when $I-y+x\in\mathcal I_1$ and an arc $x\to y$ when $I-y+x\in\mathcal I_2$. Let $X_1=\{x\notin I: I+x\in\mathcal I_1\}$ and $X_2=\{x\notin I: I+x\in\mathcal I_2\}$. A *shortest* path from $X_1$ to $X_2$ can be flipped (its outside elements enter $I$, its inside elements leave), and the result is common independent and one larger; if no path exists, the set $U$ of elements that can reach $X_2$ gives $|I|=r_1(U)+r_2(E\setminus U)$, which proves Edmonds' min formula. (Shortest matters: a path with a shortcut can break independence. The proof is in Schrijver's *Combinatorial Optimization*, Section 41.2, and Tome C2 gives the linear programming one.)

In [ ]:
def intersect(M1, M2):
    I = set()
    while True:
        out = [x for x in M1.E if x not in I]
        X1 = {x for x in out if M1.indep(list(I | {x}))}
        X2 = {x for x in out if M2.indep(list(I | {x}))}
        adj = {x: [] for x in M1.E}
        for y in I:
            for x in out:
                J = list((I - {y}) | {x})
                if M1.indep(J): adj[y].append(x)
                if M2.indep(J): adj[x].append(y)
        prev = {x: None for x in X1}; q = deque(X1); end = None
        while q:
            u = q.popleft()
            if u in X2:
                end = u; break
            for v in adj[u]:
                if v not in prev:
                    prev[v] = u; q.append(v)
        if end is None:
            # certificate: U = elements that can reach X2
            rev = {x: [] for x in M1.E}
            for u in adj:
                for v in adj[u]:
                    rev[v].append(u)
            U = set(X2); q = deque(X2)
            while q:
                v = q.popleft()
                for u in rev[v]:
                    if u not in U:
                        U.add(u); q.append(u)
            cert = M1.rank(list(U)) + M2.rank([x for x in M1.E if x not in U])
            return I, cert
        path = []
        while end is not None:
            path.append(end); end = prev[end]
        I ^= set(path)

def brute_common(M1, M2):
    F2 = set(M2.independent_sets())
    return max(len(S) for S in M1.independent_sets() if S in F2)
def brute_minformula(M1, M2):
    E = M1.E
    return min(M1.rank(list(A)) + M2.rank([x for x in E if x not in A])
               for k in range(len(E) + 1) for A in itertools.combinations(E, k))

for t in range(25):
    n = 7
    vecs1 = [tuple(random.randrange(3) for _ in range(3)) for _ in range(n)]
    G = nx.gnm_random_graph(5, n, seed=t); edges = list(G.edges())
    M2 = graphic(G)
    M1 = Matroid(range(len(edges)), lambda S, v=vecs1: rank_mod_p([v[i] for i in S], 3) == len(S))
    I, cert = intersect(M1, M2)
    assert M1.indep(list(I)) and M2.indep(list(I))
    assert len(I) == cert == brute_common(M1, M2) == brute_minformula(M1, M2)
print('25 random (linear, graphic) pairs: algorithm = brute force = min formula = certificate')

In [ ]:
for t in range(30):
    B = nx.bipartite.random_graph(8, 9, 0.25, seed=t)
    left = [u for u in B if B.nodes[u]['bipartite'] == 0]
    edges = list(B.edges())
    stars = lambda side: [[i for i, e in enumerate(edges) if side(e) == v] for v in sorted({side(e) for e in edges})]
    lside = lambda e: e[0] if e[0] in left else e[1]
    rside = lambda e: e[1] if e[0] in left else e[0]
    M1 = partition(stars(lside), [1] * len(stars(lside)))
    M2 = partition(stars(rside), [1] * len(stars(rside)))
    M1.E = M2.E = list(range(len(edges)))
    I, cert = intersect(M1, M2)
    m = nx.bipartite.maximum_matching(B, top_nodes=left) if edges else {}
    assert len(I) == len(m) // 2 == cert
print('30 random bipartite graphs: intersection of the two star partition matroids = maximum matching (Koenig)')

In [ ]:
# Rainbow spanning forest: graphic matroid  intersected with the partition matroid "each colour at most once".
G = nx.grid_2d_graph(4, 4); edges = list(G.edges())
colours = [random.randrange(9) for _ in edges]
blocks = [[i for i, c in enumerate(colours) if c == k] for k in sorted(set(colours))]
M1 = graphic(G); M2 = partition(blocks, [1] * len(blocks)); M2.E = M1.E
I, cert = intersect(M1, M2)
print(f'{len(set(colours))} colours, rank of the grid = {G.number_of_nodes() - 1}: largest rainbow forest has {len(I)} edges (certificate {cert})')
pos = {v: v for v in G}
cmap = plt.get_cmap('tab10')
nx.draw_networkx_edges(G, pos, edgelist=edges, edge_color=[cmap(c) for c in colours], width=1, style='dotted')
nx.draw_networkx_edges(G, pos, edgelist=[edges[i] for i in I], edge_color=[cmap(colours[i]) for i in I], width=4)
nx.draw_networkx_nodes(G, pos, node_size=40, node_color='k')
plt.axis('equal'); plt.axis('off'); plt.title('a largest rainbow forest (thick)'); plt.show()

With $9$ colours a forest can use at most $9$ edges, below the rank $15$ of the grid, so the colour constraint binds, and the certificate shows the forest found cannot be beaten. The same algorithm, given the right pair of matroids, finds matchings, transversals, rainbow trees, and (with weights, in Tome C2) assignments: two matroids are the frontier of the good theory, since three already contain the Hamiltonian path problem.